# 보호 노이즈 효과 확인 (Colab)
런타임 > 런타임 유형 변경 > **T4 GPU** 선택 후 위에서부터 실행하세요.

이 노트북은 실험용이며 Signal 서비스와 연결되어 있지 않습니다.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors fastapi uvicorn python-multipart pytest httpx

In [ ]:
# signal-protection-lab 폴더를 zip으로 압축해서 업로드
from google.colab import files
up = files.upload()
!unzip -oq signal-protection-lab.zip
%cd signal-protection-lab

In [ ]:
# 테스트 이미지를 업로드 (본인 사진이나 공개 샘플 사진 3~5장 권장, 타인 사진 금지)
up = files.upload()
import os
photos = list(up.keys()); photos

In [ ]:
# 수준별 보호 처리 (VAE 모델은 첫 실행 때 자동으로 내려받음)
for p in photos:
    for lv in ['LIGHT','NORMAL','STRONG']:
        !python cli.py "{p}" "prot_{lv}_{os.path.splitext(p)[0]}.png" --level {lv}

In [ ]:
# 원본 vs 보호본을 Stable Diffusion img2img에 넣어 비교
for p in photos:
    stem = os.path.splitext(p)[0]
    !python compare_sd.py "{p}" "prot_NORMAL_{stem}.png" --out "grid_NORMAL_{stem}.png"

from IPython.display import Image, display
import glob
for g in sorted(glob.glob('grid_*.png')):
    print(g); display(Image(g))

## 결과 읽는 법
- 왼쪽부터 **원본 | 보호본 | 원본 편집 결과 | 보호본 편집 결과** 입니다.
- 오른쪽 두 장을 비교해 보호본 편집 결과가 눈에 띄게 깨지거나 회색 얼룩이 생기면 효과가 있는 것입니다.
- 차이가 없다면 "이 조건에서는 효과를 확인하지 못했다"고 기록하세요. 수준(LIGHT/NORMAL/STRONG)과 --strength를 바꿔 가며 몇 번 더 시도해 볼 수 있습니다.
- 이 결과는 Stable Diffusion 1.5 img2img에 한정된 것이며, 얼굴 교체형 도구나 JPEG 재압축에는 일반화할 수 없습니다.